# Notebook 4 — Pré-processamento e Adaptação dos Dados
**Projeto:** Predição de Evasão Escolar em Pernambuco  
**Disciplina:** Aprendizagem de Máquina | **Entrega:** 11/05/2026
---

## 1. Objetivos

1. Selecionar features relevantes, removendo redundâncias e multicolinearidade  
2. Imputar valores ausentes com estratégia baseada em grupos  
3. Codificar variáveis categóricas (One-Hot Encoding)  
4. Escalar features numéricas (StandardScaler)  
5. Tratar o desbalanceamento da variável-alvo (SMOTE)  
6. Construir o pipeline final e salvar o dataset pronto para modelagem

## 2. Importações

In [2]:
from pathlib import Path
import os

# Garante que o diretório de trabalho é sempre a raiz do projeto
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print(f'Diretório de trabalho: {PROJECT_ROOT}')

Diretório de trabalho: /Users/joaogui/Downloads/1VA_Aprendizado_Maquina


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# SMOTE para balanceamento
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_DISPONIVEL = True
except ImportError:
    print("imblearn nao instalado. Executar: pip install imbalanced-learn")
    SMOTE_DISPONIVEL = False

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

AZUL  = '#378ADD'
CORAL = '#D85A30'
CINZA = '#888780'
print("Importacoes concluidas.")

Importacoes concluidas.


## 3. Carregamento e Filtragem

In [4]:
df_ufs = pd.read_csv('data/processed/dataset_integrado.csv')

print(f"Dataset municipal: {df_ufs.shape}")
print(f"Municípios: {df_ufs['cod_municipio'].nunique()} | UFs: {df_ufs['uf'].nunique()}")
print(f"Distribuicao alvo: {df_ufs['risco_evasao'].value_counts().to_dict()}")
print(f"Balanceamento: {df_ufs['risco_evasao'].mean()*100:.1f}% positivos")

Dataset municipal: (50140, 84)
Municípios: 5570 | UFs: 27
Distribuicao alvo: {0: 37715, 1: 12425}
Balanceamento: 24.8% positivos


## 4. Seleção de Features

Quatro critérios guiam a seleção:

1. **Leakage com a variável-alvo:** `risco_evasao = 1` se `abandono_geral > P75`. Qualquer coluna de abandono (`abandono_fund_total`, `abandono_med_total`, etc.) revela diretamente o alvo — **todas removidas**.  
2. **Redundância estrutural:** `aprovação + reprovação + abandono = 100` por definição. Com o abandono já removido, a aprovação também é excluída por ser derivável da reprovação — ficamos apenas com reprovação.  
3. **Alta correlação entre features de infra (>0.75):** 39 pares identificados na EDA. Mantemos uma feature por grupo temático.  
4. **Missing alto (>10%):** features por série/ano específico têm até 14.8% de NaN, concentrado em combinações raras. Preferimos os totais por etapa que têm menor missing.

In [5]:
# ── Features de reprovacao ────────────────────────────────────────────────
# Aprovacao EXCLUIDA: aprov + reprov + abandono = 100 (redundante)
# Abandono  EXCLUIDO: leakage — a variavel-alvo e derivada do abandono
FEAT_REPROV = [
    'reprov_fund_total',       # reprovacao EF total
    'reprov_fund_anos_finais', # anos finais EF (mais critico)
    'reprov_med_total',        # reprovacao EM total
    'reprov_med_1serie',       # 1a serie EM (maior gargalo)
]

# ── Features de infraestrutura (uma por grupo tematico) ───────────────────
# Grupo digital:     pct_internet (exclui pct_computador — r=0.91)
# Grupo pedagogico:  pct_biblioteca (exclui pct_sala_leitura — r=0.92)
#                    pct_lab_informatica
#                    pct_quadra
# Grupo basico:      pct_agua_potavel (exclui pct_energia_rede — r=0.80)
#                    pct_sem_esgoto
# Grupo acesso:      pct_sem_acessibilidade (exclui pct_banheiro_acessivel — r=0.87)
# Outros:            pct_alimentacao, qt_salas_media
FEAT_INFRA = [
    'pct_internet',
    'pct_biblioteca',
    'pct_lab_informatica',
    'pct_quadra',
    'pct_agua_potavel',
    'pct_sem_esgoto',
    'pct_sem_acessibilidade',
    'pct_alimentacao',
    'qt_salas_media',
]

# ── Categoricas ───────────────────────────────────────────────────────────
FEAT_CAT = ['localizacao', 'dependencia_adm']

TODAS_FEATURES = FEAT_REPROV + FEAT_INFRA + FEAT_CAT
ALVO = 'risco_evasao'

print(f"Features selecionadas: {len(TODAS_FEATURES)}")
print(f"  Reprovacao:    {len(FEAT_REPROV)}")
print(f"  Infraestrutura:{len(FEAT_INFRA)}")
print(f"  Categoricas:   {len(FEAT_CAT)}")
print(f"\nREMOVIDAS por leakage: abandono_fund_total, abandono_fund_anos_finais, abandono_med_total")
print(f"REMOVIDAS por redundancia: todas as colunas aprov_*")

# Verificar missing no conjunto final
X_raw = df_ufs[TODAS_FEATURES]
y = df_ufs[ALVO]
miss = (X_raw.isnull().sum() / len(X_raw) * 100).round(1)
print(f"\nMissing por feature:")
print(miss[miss > 0].sort_values(ascending=False).to_string())
print(f"\nFeatures sem missing: {(miss == 0).sum()} de {len(TODAS_FEATURES)}")

Features selecionadas: 15
  Reprovacao:    4
  Infraestrutura:9
  Categoricas:   2

REMOVIDAS por leakage: abandono_fund_total, abandono_fund_anos_finais, abandono_med_total
REMOVIDAS por redundancia: todas as colunas aprov_*

Missing por feature:
reprov_med_1serie          41.1
reprov_med_total           40.5
reprov_fund_anos_finais    24.3
reprov_fund_total          10.5

Features sem missing: 11 de 15


## 5. Imputação de Valores Ausentes

In [6]:
# ── Estrategia: mediana por grupo (dependencia_adm + localizacao) ─────────
# Justificativa: uma escola Estadual Rural de PE nao deve receber a mediana
# global — deve receber a mediana de outras escolas Estaduais Rurais.
# Isso preserva a estrutura dos dados.

FEAT_NUM = FEAT_REPROV + FEAT_INFRA
df_proc = df_ufs[TODAS_FEATURES + [ALVO]].copy()

for col in FEAT_NUM:
    if df_proc[col].isna().sum() > 0:
        mediana_grupo = df_proc.groupby(['dependencia_adm','localizacao'])[col].transform('median')
        mediana_global = df_proc[col].median()
        df_proc[col] = df_proc[col].fillna(mediana_grupo).fillna(mediana_global)

# Verificar resultado
miss_pos = (df_proc[FEAT_NUM].isnull().sum()).sum()
print(f"Missing apos imputacao por grupo: {miss_pos} celulas")
print(f"Shape mantido: {df_proc.shape}")

# Visualizar impacto da imputacao em uma coluna exemplo
col_ex = 'reprov_med_total'
antes = df_ufs[col_ex].isna().sum()
depois = df_proc[col_ex].isna().sum()
print(f"\nExemplo — {col_ex}: {antes} NaN antes -> {depois} NaN depois")

Missing apos imputacao por grupo: 0 celulas
Shape mantido: (50140, 16)

Exemplo — reprov_med_total: 20330 NaN antes -> 0 NaN depois


## 6. Encoding das Variáveis Categóricas

In [7]:
# ── One-Hot Encoding (sem drop_first para manter interpretabilidade) ──────
# drop='first' causaria problemas de interpretacao no SHAP — mantemos todas
# as dummies e deixamos o modelo lidar com a redundancia.
# Para modelos baseados em arvore (RF, XGBoost) isso e aceitavel.
# Para Regressao Logistica, usamos drop='first'.

df_encoded = pd.get_dummies(df_proc, columns=FEAT_CAT, drop_first=False)

# Identificar novas colunas geradas
cols_dummies = [c for c in df_encoded.columns if c.startswith(('localizacao_','dependencia_adm_'))]
print(f"Dummies geradas: {cols_dummies}")
print(f"Shape apos encoding: {df_encoded.shape}")

# Atualizar lista de features numericas e categoricas
FEAT_FINAL_NUM = FEAT_NUM
FEAT_FINAL_CAT_ENC = cols_dummies
TODAS_FEAT_FINAL = FEAT_FINAL_NUM + FEAT_FINAL_CAT_ENC

print(f"\nFeatures finais para o modelo: {len(TODAS_FEAT_FINAL)}")

Dummies geradas: ['localizacao_Rural', 'localizacao_Total', 'localizacao_Urbana', 'dependencia_adm_Estadual', 'dependencia_adm_Federal', 'dependencia_adm_Municipal', 'dependencia_adm_Privada', 'dependencia_adm_Total']
Shape apos encoding: (50140, 22)

Features finais para o modelo: 21


## 7. Escalonamento

In [8]:
from sklearn.preprocessing import StandardScaler

X = df_encoded[TODAS_FEAT_FINAL].copy()
y = df_encoded[ALVO].copy()

# Escalar apenas as features numericas (dummies ja estao em 0/1)
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X[FEAT_FINAL_NUM])
X_num_df = pd.DataFrame(X_num_scaled, columns=FEAT_FINAL_NUM, index=X.index)

# Montar X final
X_final = pd.concat([X_num_df, X[FEAT_FINAL_CAT_ENC]], axis=1)

print(f"Shape X final: {X_final.shape}")
print(f"Shape y: {y.shape}")
print(f"\nApos escalonamento — estatisticas das features numericas:")
print(X_final[FEAT_FINAL_NUM].describe().round(2).T[['mean','std','min','max']].to_string())

Shape X final: (50140, 21)
Shape y: (50140,)

Apos escalonamento — estatisticas das features numericas:
                         mean  std   min     max
reprov_fund_total         0.0  1.0 -0.85   11.50
reprov_fund_anos_finais  -0.0  1.0 -0.94    8.30
reprov_med_total          0.0  1.0 -0.94    9.15
reprov_med_1serie        -0.0  1.0 -0.93    8.22
pct_internet             -0.0  1.0 -6.18    0.32
pct_biblioteca           -0.0  1.0 -1.28    1.36
pct_lab_informatica      -0.0  1.0 -1.09    1.66
pct_quadra                0.0  1.0 -1.30    1.58
pct_agua_potavel         -0.0  1.0 -9.35    0.21
pct_sem_esgoto            0.0  1.0 -0.19   10.38
pct_sem_acessibilidade   -0.0  1.0 -0.72    2.71
pct_alimentacao           0.0  1.0 -3.39    0.42
qt_salas_media            0.0  1.0 -1.17  138.13


## 8. Tratamento do Desbalanceamento

In [9]:
# ── Analise do desbalanceamento ────────────────────────────────────────────
n_neg = (y == 0).sum()
n_pos = (y == 1).sum()
print(f"Classe 0 (baixo risco): {n_neg} ({n_neg/len(y)*100:.1f}%)")
print(f"Classe 1 (alto risco):  {n_pos} ({n_pos/len(y)*100:.1f}%)")
print(f"Razao de desbalanceamento: {n_neg/n_pos:.1f}:1")

# ── Estrategia adotada ─────────────────────────────────────────────────────
# Razao ~3:1 e moderada. Usaremos class_weight='balanced' nos modelos
# (equivale a penalizar mais os erros na classe minoritaria).
# SMOTE nao e necessario: com ~50k instancias, o modelo tem dados suficientes
# para aprender a classe minoritaria sem sintese artificial.

print("\nEstrategia: class_weight='balanced' nos modelos (default)")
print("Justificativa: razao moderada (~3:1); dataset municipal tem dados suficientes")

Classe 0 (baixo risco): 37715 (75.2%)
Classe 1 (alto risco):  12425 (24.8%)
Razao de desbalanceamento: 3.0:1

Estrategia: class_weight='balanced' nos modelos (default)
Justificativa: razao moderada (~3:1); dataset municipal tem dados suficientes


## 9. Divisão dos Dados

In [10]:
from sklearn.model_selection import train_test_split

# Identificar indices de PE para avaliacao focada
idx_pe = df_ufs[df_ufs['uf'] == 'PE'].index.tolist()
idx_outros = [i for i in range(len(X_final)) if i not in idx_pe]

print(f"Instancias de PE: {len(idx_pe)} ({df_ufs.loc[df_ufs['uf']=='PE','cod_municipio'].nunique()} municípios)")
print(f"Instancias de outras UFs: {len(idx_outros)}")

# Divisao principal: treino/teste estratificada — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"\nTreino: {X_train.shape[0]} instancias ({y_train.sum()} positivos, {y_train.mean()*100:.1f}%)")
print(f"Teste:  {X_test.shape[0]} instancias ({y_test.sum()} positivos, {y_test.mean()*100:.1f}%)")
print(f"Estratificacao preservada: OK")

# Validacao cruzada: StratifiedKFold k=5
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"\nValidacao cruzada: StratifiedKFold k=5")
print(f"Ganho vs dataset por UF: {len(X_final)} instancias ({len(X_final)//398}x mais dados)")

Instancias de PE: 1912 (185 municípios)
Instancias de outras UFs: 48228

Treino: 40112 instancias (9940 positivos, 24.8%)
Teste:  10028 instancias (2485 positivos, 24.8%)
Estratificacao preservada: OK

Validacao cruzada: StratifiedKFold k=5
Ganho vs dataset por UF: 50140 instancias (125x mais dados)


## 10. Baseline — DummyClassifier

In [11]:
from sklearn.metrics import classification_report, roc_auc_score, recall_score

# Baseline: sempre prediz a classe majoritaria
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("BASELINE — DummyClassifier (most_frequent):")
print(classification_report(y_test, y_pred_dummy, target_names=['Baixo risco','Alto risco']))
print(f"ROC-AUC: {roc_auc_score(y_test, dummy.predict_proba(X_test)[:,1]):.3f}")
print(f"Recall classe 1: {recall_score(y_test, y_pred_dummy):.3f}")
print()
print("Interpretacao:")
print("  Accuracy de 75% sem aprender nada — confirma que Accuracy nao e boa metrica aqui")
print("  ROC-AUC = 0.5 e Recall = 0.0 sao o piso absoluto que qualquer modelo deve superar")

BASELINE — DummyClassifier (most_frequent):
              precision    recall  f1-score   support

 Baixo risco       0.75      1.00      0.86      7543
  Alto risco       0.00      0.00      0.00      2485

    accuracy                           0.75     10028
   macro avg       0.38      0.50      0.43     10028
weighted avg       0.57      0.75      0.65     10028

ROC-AUC: 0.500
Recall classe 1: 0.000

Interpretacao:
  Accuracy de 75% sem aprender nada — confirma que Accuracy nao e boa metrica aqui
  ROC-AUC = 0.5 e Recall = 0.0 sao o piso absoluto que qualquer modelo deve superar


## 11. Verificação do Pipeline com Regressão Logística

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, f1_score

# Teste rapido para validar que o pipeline esta funcional
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# Validacao cruzada no conjunto de treino
cv_results = cross_validate(
    lr, X_train, y_train,
    cv=skf,
    scoring=['roc_auc','recall','f1'],
    return_train_score=False
)

print("Regressao Logistica — validacao cruzada (treino):")
print(f"  ROC-AUC: {cv_results['test_roc_auc'].mean():.3f} (+/- {cv_results['test_roc_auc'].std():.3f})")
print(f"  Recall:  {cv_results['test_recall'].mean():.3f} (+/- {cv_results['test_recall'].std():.3f})")
print(f"  F1:      {cv_results['test_f1'].mean():.3f} (+/- {cv_results['test_f1'].std():.3f})")
print()
print("Pipeline validado — pronto para experimentacao completa no Notebook 5")

Regressao Logistica — validacao cruzada (treino):
  ROC-AUC: 0.825 (+/- 0.007)
  Recall:  0.801 (+/- 0.005)
  F1:      0.598 (+/- 0.007)

Pipeline validado — pronto para experimentacao completa no Notebook 5


## 12. Salvamento dos Dados Processados

In [13]:
import joblib

# Salvar X_train, X_test, y_train, y_test e metadados
X_train.to_csv('data/processed/X_train.csv', index=False)
X_test.to_csv('data/processed/X_test.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

# Salvar lista de features e scaler para reprodutibilidade
import json
metadados = {
    'features_numericas': FEAT_FINAL_NUM,
    'features_categoricas_enc': FEAT_FINAL_CAT_ENC,
    'todas_features': TODAS_FEAT_FINAL,
    'alvo': ALVO,
    'n_treino': len(X_train),
    'n_teste': len(X_test),
    'limiar_risco': float(df_ufs['abandono_geral'].quantile(0.75)),
    'balanceamento_treino': float(y_train.mean()),
    'granularidade': 'municipio',
    'n_municipios': int(df_ufs['cod_municipio'].nunique()),
}
with open('models/metadados_pipeline.json', 'w') as f:
    json.dump(metadados, f, indent=2, ensure_ascii=False)

joblib.dump(scaler, 'models/scaler.joblib')

print("Arquivos salvos:")
print("  X_train.csv, X_test.csv — features processadas")
print("  y_train.csv, y_test.csv — variavel-alvo")
print("  metadados_pipeline.json  — configuracao do pipeline")
print("  scaler.joblib            — scaler para reproducibilidade")
print(f"\nResumo final:")
print(f"  Features: {len(TODAS_FEAT_FINAL)}")
print(f"  Treino:   {len(X_train)} instancias")
print(f"  Teste:    {len(X_test)} instancias")
print(f"  Alvo:     {y_train.mean()*100:.1f}% positivos no treino")

Arquivos salvos:
  X_train.csv, X_test.csv — features processadas
  y_train.csv, y_test.csv — variavel-alvo
  metadados_pipeline.json  — configuracao do pipeline
  scaler.joblib            — scaler para reproducibilidade

Resumo final:
  Features: 21
  Treino:   40112 instancias
  Teste:    10028 instancias
  Alvo:     24.8% positivos no treino


## 13. Resumo das Decisões de Pré-processamento

| # | Etapa | Decisão | Justificativa |
|---|-------|---------|---------------|
| 1 | Seleção — **leakage** | Remover todas as colunas `abandono_*` | `risco_evasao` é derivado do `abandono_geral` → usar abandono como feature revela o alvo diretamente |
| 2 | Seleção — redundância estrutural | Remover `aprov_*` | `aprov + reprov + abandono = 100` → aprovação é derivável da reprovação |
| 3 | Seleção — infra | Manter 1 feature por grupo temático | 39 pares com r > 0.75 identificados na EDA |
| 4 | Seleção — missing | Preferir totais por etapa | Menor missing que colunas por série específica |
| 5 | Imputação | Mediana por grupo (dep × loc) | Preserva estrutura dos dados; mais preciso que mediana global |
| 6 | Encoding | One-Hot sem drop_first | Preserva interpretabilidade na análise de feature importance |
| 7 | Escalonamento | StandardScaler nas numéricas | Necessário para Logística, SVM e KNN; inócuo para árvores |
| 8 | Desbalanceamento | `class_weight='balanced'` | Razão ~3:1; dataset municipal tem dados suficientes sem SMOTE |
| 9 | Divisão | 80/20 estratificada + StratifiedKFold k=5 | ~40k treino / 10k teste preservando proporção das classes |
| 10 | Baseline | DummyClassifier (most_frequent) | ROC-AUC=0.5, Recall=0.0 — piso que qualquer modelo deve superar |